# Лабораторная работа №1: Подготовка и первичный анализ данных для проверки бизнес-гипотез

#### __1. Формулировка бизнес-задачи__

__Контекст:__ Данные представляют собой информацию о задачах (collection), выполняемых в крупном кластере Google (Google Cluster Data).

__Бизнес-задача:__ Крупные компании, такие как Google, эксплуатируют огромные кластеры серверов. Эффективное управление ресурсами этих кластеров (CPU, память) критически важно для минимизации затрат на оборудование и электроэнергию, а также для обеспечения стабильной работы сервисов. Цель — понять закономерности потребления ресурсов задачами (экземплярами) в кластере.

#### __2. Гипотезы о факторах, влияющих на решение задачи__

- __H1:__ Существует прямая зависимость между средним (`average_usage`) и максимальным (`maximum_usage`) потреблением ресурсов. Если среднее использование высоко, то и пиковые значения будут высокими.
- __H2:__ Наличие страничного кэша (`page_cache_memory`) снижает максимальное использование оперативной памяти (`maximum_usage.memory`), так как часть данных уже кэширована.
- __H3:__ Разные типы задач (заданные полем `collection_type`) имеют различные профили потребления CPU. Например, сервисы (`type=0`) могут требовать меньше ресурсов, чем пакетные задания (`type=1`).
- __H4:__ Высокие значения `cycles_per_instruction` (CPI) могут коррелировать с низкой эффективностью использования CPU, что отражается на `average_usage.cpus`.

#### __3. Определение необходимых и доступных данных__
 
 __Необходимые данные:__
- `average_usage`: Среднее использование CPU и памяти
- `maximum_usage`: Максимальное использование CPU и памяти
- `page_cache_memory`: для проверки влияния кэша на производительность
- `cycles_per_instruction`: Количество тактов на инструкцию (CPI)
- `memory_accesses_per_instruction`: Количество обращений к памяти на инструкцию (MAP)
- `cpu_usage_distribution`: Распределение использования CPU во времени
- `assigned_memory`: Выделенный объем памяти
- `collection_type`: для сравнения профилей разных типов задач
- `cpu_usage_distribution`: Список значений (доли времени), представляющий гистограмму использования CPU
- `tail_cpu_usage_distribution`: Список значений (доли времени), представляющий детализацию хвоста распределения (пиковых нагрузок).
 
__Доступные данные:__ Используется датасет Dataset.csv, содержащий выгрузку из системы управления кластером Google (Google Cluster Data). Данные включают информацию о задачах, их потреблении ресурсов, метрики производительности, а также временные метки выполнения.

##### __Номинальная шкала:__
- `collection_id`: Уникальный идентификатор набора задач
- `machine_id`: Идентификатор машины в кластере
- `alloc_collection_id`: Идентификатор родительской коллекции
- `instance_index`: Индекс экземпляра
- `alloc_instance_index`: Индекс выделенного экземпляра

##### __Порядковая шкала:__
- `collection_type`: Типы коллекций (0,1,2,3)

##### __Интервальная шкала:__
- `start_time`: Временная метка начала
- `end_time`: Временная метка окончания
- `cycles_per_instruction`: Количество тактов на инструкцию
- `memory_accesses_per_instruction`: Количество обращений к памяти на инструкцию

##### __Относительная шкала:__
- `average_usage`: Среднее использование CPU и памяти
- `maximum_usage`: Максимальное использование CPU и памяти
- `random_sample_usage`: Случайная выборка использования CPU
- `page_cache_memory`: Доля памяти под страничный кэш
- `assigned_memory`: Выделенный объем памяти
- `sample_rate`: Частота дискретизации
- `cpu_usage_distribution`: Список значений (доли времени), представляющий гистограмму использования CPU
- `tail_cpu_usage_distribution`: Список значений (доли времени), представляющий детализацию хвоста распределения (пиковых нагрузок) 

#### __4. Загрузка и подготовка набора данных__

In [11]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
import ast
import numpy as np
warnings.filterwarnings('ignore')

# Настройка отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Загрузка данных
df = pd.read_csv('Data\Dataset.csv', sep = ',', encoding='latin1')

In [12]:
# размер датасета и наличие пропусков
print("\nРазмер датасета:", df.shape)

print(df.isnull().sum())


Размер датасета: (498, 18)
start_time                         0
end_time                           0
collection_id                      0
instance_index                     0
machine_id                         0
alloc_collection_id                0
alloc_instance_index               0
collection_type                    0
average_usage                      0
maximum_usage                      0
random_sample_usage                0
assigned_memory                    0
page_cache_memory                  0
cycles_per_instruction             0
memory_accesses_per_instruction    0
sample_rate                        0
cpu_usage_distribution             0
tail_cpu_usage_distribution        0
dtype: int64


In [13]:
# Проверка дубликатов
print(f"Количество дубликатов: {df.duplicated().sum()}")

Количество дубликатов: 0


In [14]:
# осмотр типов данных
print(df.dtypes)

start_time                           int64
end_time                             int64
collection_id                        int64
instance_index                       int64
machine_id                           int64
alloc_collection_id                  int64
alloc_instance_index                 int64
collection_type                      int64
average_usage                       object
maximum_usage                       object
random_sample_usage                 object
assigned_memory                      int64
page_cache_memory                  float64
cycles_per_instruction             float64
memory_accesses_per_instruction    float64
sample_rate                        float64
cpu_usage_distribution              object
tail_cpu_usage_distribution         object
dtype: object


#### __5. Очистка данных и преобразование__

In [15]:
# копия для очистки
df_clean = df.copy()

In [16]:
# удаление дубликатов
df_clean.drop_duplicates()

,start_time,end_time,collection_id,instance_index,machine_id,alloc_collection_id,alloc_instance_index,collection_type,average_usage,maximum_usage,random_sample_usage,assigned_memory,page_cache_memory,cycles_per_instruction,memory_accesses_per_instruction,sample_rate,cpu_usage_distribution,tail_cpu_usage_distribution
0,1838400000000,1838700000000,330587238433,111,23624491139,330587160469,111,0,"{'cpus': 0.008392333984375, 'memory': 0.009475...","{'cpus': 0.04791259765625, 'memory': 0.0193786...",{'cpus': 0.016632080078125},0,0.003220,0.899244,0.001512,0.996667,"[0.0004711151123046875, 0.0006809234619140625,...","[0.022308349609375, 0.023681640625, 0.02420043..."
1,1839000000000,1839300000000,330587238433,111,23624491139,330587160469,111,0,"{'cpus': 0.027099609375, 'memory': 0.027618408...","{'cpus': 0.0736083984375, 'memory': 0.02795410...",{'cpus': 0.019683837890625},0,0.001808,0.899244,0.001512,1.000000,"[0.0088043212890625, 0.016357421875, 0.0186157...","[0.04290771484375, 0.0440673828125, 0.04663085..."
2,942000000000,942300000000,330587238433,111,23624491139,330587160469,111,0,"{'cpus': 0.024932861328125, 'memory': 0.012466...","{'cpus': 0.11767578125, 'memory': 0.019287109375}",{'cpus': 0.00860595703125},0,0.002392,1.028699,0.003364,1.000000,"[5.435943603515625e-05, 0.00020122528076171875...","[0.052001953125, 0.052490234375, 0.052734375, ..."
3,941700000000,942000000000,330587238433,111,23624491139,330587160469,111,0,"{'cpus': 0.019134521484375, 'memory': 0.010955...","{'cpus': 0.1209716796875, 'memory': 0.01931762...",{'cpus': 0.00020503997802734375},0,0.001934,1.075302,0.003459,1.000000,"[4.482269287109375e-05, 7.915496826171875e-05,...","[0.04559326171875, 0.04620361328125, 0.0468139..."
4,1837800000000,1838100000000,330587238433,111,23624491139,330587160469,111,0,"{'cpus': 0.001163482666015625, 'memory': 0.000...","{'cpus': 0.0110626220703125, 'memory': 0.00195...",{'cpus': 0.0009307861328125},0,0.000813,0.899244,0.001512,1.000000,"[8.7738037109375e-05, 0.00010204315185546875, ...","[0.00592803955078125, 0.00677490234375, 0.0073..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493,175500000000,175800000000,330587238433,0,345111144299,330587160469,0,0,"{'cpus': 0.031494140625, 'memory': 0.022583007...","{'cpus': 0.0892333984375, 'memory': 0.02371215...",{'cpus': 0.001407623291015625},0,0.002480,0.837627,0.006004,0.996667,"[0.0004711151123046875, 0.0095672607421875, 0....","[0.04351806640625, 0.04388427734375, 0.0442504..."
494,177600000000,177900000000,330587238433,0,345111144299,330587160469,0,0,"{'cpus': 0.03668212890625, 'memory': 0.0163269...","{'cpus': 0.189208984375, 'memory': 0.016876220...",{'cpus': 0.02935791015625},0,0.000582,1.685948,0.009581,0.996667,"[0.0084991455078125, 0.021331787109375, 0.0246...","[0.071533203125, 0.0743408203125, 0.0786132812..."
495,175200000000,175500000000,330587238433,0,345111144299,330587160469,0,0,"{'cpus': 0.015533447265625, 'memory': 0.012298...","{'cpus': 0.34716796875, 'memory': 0.0239562988...",{'cpus': 0.0333251953125},0,0.002056,0.864440,0.006975,0.940000,"[0.0001049041748046875, 0.0006084442138671875,...","[0.03814697265625, 0.0391845703125, 0.03997802..."
496,784200000000,784500000000,330587238433,1667,376022651559,330587160469,1667,0,"{'cpus': 0.01123046875, 'memory': 0.0120849609...","{'cpus': 0.0633544921875, 'memory': 0.01849365...",{'cpus': 0.0302734375},0,0.001368,0.693165,0.006117,0.996667,"[0.00018310546875, 0.00043964385986328125, 0.0...","[0.03802490234375, 0.03857421875, 0.0391235351..."


In [17]:
# Удалил 7 столбцов т.к они бесполезны для анализа
cols_to_drop = ['instance_index', 'collection_id', 'machine_id', 'alloc_collection_id', 'alloc_instance_index', 'random_sample_usage', 'assigned_memory', 'sample_rate'] 
df_clean = df_clean.drop(cols_to_drop, axis=1, errors='ignore')

In [18]:
# преобразую признаки start_time и end_time в duration (длительность задачи в секундах)
df['duration'] = (df['end_time'] - df['start_time']) / 1000000000  # делим на 1e9 для перевода в секунды

# Удалил start_time и end_time
cols_to_drop = ['start_time', 'end_time'] 
df_clean = df_clean.drop(cols_to_drop, axis=1, errors='ignore')

columns_order = ['duration'] + [col for col in df_clean.columns if col != 'duration']

df_clean = df[columns_order]

In [19]:
# Функция для парсинга словаря и извлечения значений
def parse_usage_dict(usage_str):
    try:
        # Преобразуем строку в словарь
        usage_dict = ast.literal_eval(usage_str)
        # Извлекаем значения для CPU и memory
        cpus = usage_dict.get('cpus', None)
        memory = usage_dict.get('memory', None)
        return pd.Series([cpus, memory])
    except:
        return pd.Series([None, None])

# Применяем функцию к столбцам average_usage и maximum_usage
df_clean[['avg_cpus', 'avg_memory']] = df['average_usage'].apply(parse_usage_dict)
df_clean[['max_cpus', 'max_memory']] = df['maximum_usage'].apply(parse_usage_dict)

# переупорядочиваю столбцы
cols = []
for col in df_clean.columns:
    cols.append(col)
    if col == 'average_usage':
        cols.append('avg_cpus')
        cols.append('avg_memory')
    if col == 'maximum_usage':
        cols.append('max_cpus')
        cols.append('max_memory')
        
cols = list(dict.fromkeys(cols))  # сохраняет порядок, убирая дубликаты        
        
df_clean = df_clean[cols]

# Удаляем исходные столбцы (опционально)
df_clean = df_clean.drop(['average_usage', 'maximum_usage'], axis=1)

In [20]:
# размер датасета и наличие пропусков
print("\nРазмер датасета:", df_clean.shape)

missing_data = df_clean.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({'Количество пропусков': missing_data, 'Доля пропусков (%)': missing_percent})
print(missing_df)


Размер датасета: (498, 11)
                                 Количество пропусков  Доля пропусков (%)
duration                                            0                 0.0
collection_type                                     0                 0.0
avg_cpus                                            0                 0.0
avg_memory                                          0                 0.0
max_cpus                                            0                 0.0
max_memory                                          0                 0.0
page_cache_memory                                   0                 0.0
cycles_per_instruction                              0                 0.0
memory_accesses_per_instruction                     0                 0.0
cpu_usage_distribution                              0                 0.0
tail_cpu_usage_distribution                         0                 0.0


#### __6. Масштабирование признаков__

In [21]:
# Инициализируем масштабаторы
scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()

df_clean['memory_accesses_per_instruction_std'] = scaler_standard.fit_transform(df_clean[['memory_accesses_per_instruction']])
df_clean['cycles_per_instruction_norm'] = scaler_minmax.fit_transform(df_clean[['cycles_per_instruction']])

In [22]:
# переупорядочиваю столбцы
cols = []
for col in df_clean.columns:
    cols.append(col)
    if col == 'memory_accesses_per_instruction':
        cols.append('memory_accesses_per_instruction_std')
    if col == 'cycles_per_instruction':
        cols.append('cycles_per_instruction_norm')

cols = list(dict.fromkeys(cols))  # сохраняет порядок, убирая дубликаты

df_clean = df_clean[cols]

df_clean.head(5)

,duration,collection_type,avg_cpus,avg_memory,max_cpus,max_memory,page_cache_memory,cycles_per_instruction,cycles_per_instruction_norm,memory_accesses_per_instruction,memory_accesses_per_instruction_std,cpu_usage_distribution,tail_cpu_usage_distribution
0,0.3,0,0.008392,0.009476,0.047913,0.019379,0.003220,0.899244,0.063542,0.001512,-0.788944,"[0.0004711151123046875, 0.0006809234619140625,...","[0.022308349609375, 0.023681640625, 0.02420043..."
1,0.3,0,0.027100,0.027618,0.073608,0.027954,0.001808,0.899244,0.063542,0.001512,-0.788944,"[0.0088043212890625, 0.016357421875, 0.0186157...","[0.04290771484375, 0.0440673828125, 0.04663085..."
2,0.3,0,0.024933,0.012466,0.117676,0.019287,0.002392,1.028699,0.087088,0.003364,-0.245127,"[5.435943603515625e-05, 0.00020122528076171875...","[0.052001953125, 0.052490234375, 0.052734375, ..."
3,0.3,0,0.019135,0.010956,0.120972,0.019318,0.001934,1.075302,0.095564,0.003459,-0.217250,"[4.482269287109375e-05, 7.915496826171875e-05,...","[0.04559326171875, 0.04620361328125, 0.0468139..."
4,0.3,0,0.001163,0.000959,0.011063,0.001957,0.000813,0.899244,0.063542,0.001512,-0.788944,"[8.7738037109375e-05, 0.00010204315185546875, ...","[0.00592803955078125, 0.00677490234375, 0.0073..."


#### __Сохранение датасета__

In [23]:
df_clean.to_csv('Data\Dataset_1laba.csv', index = False, encoding='utf-8')

# Ответы на вопросы по теме:

### 1. Перечислите практические приемы решения проблемы с недостающими данными.
- удаление строк
- статистическое восполнение
- восполнение на основе модели
- использование методов машинного обучения

### 2. Зачем признаки нужно приводить к одному масштабу измерений?
- признаки с большими значениями доминируют над признаками с малыми значениями при вычислении расстояний.
- признаки в разных масштабах дают разные по величине градиенты, что замедляет сходимость.
- штрафы применяются неравномерно к признакам разного масштаба.
- в линейных моделях веса показывают реальную важность признаков только после масштабирования.

### 3. Приведите примеры алгоритмов инвариантных к масштабу измерений.
Древовидные модели:
- Decision Trees (деревья решений)
- Random Forest (случайный лес)
- Gradient Boosting (градиентный бустинг)

Наивный Байес (Naive Bayes)

Ранговые методы

### 4. Какие подходы к приведению признаков к одному и тому же масштабу вы знаете?
- Нормализация 
- Стандартизация 

### 5. Какие решения, направленные на сокращение ошибки, обобщения вы знаете?
- Регуляризация
- Кросс-валидация
- Упрощение модели
- Ансамблевые методы
- Ранняя остановка
- Увеличение данных